# 02 Giải Đáp MCQ

Notebook này giải 10 câu hỏi trắc nghiệm trong Phần 1. Mỗi câu hỏi bao gồm nội dung đề, logic tính toán, bảng kiểm chứng và đáp án cuối cùng A/B/C/D. Bảng tổng hợp đáp án được xuất ra `artifacts/mcq/mcq_answers.csv`.

## 1. Cài Đặt Và Tải Dữ Liệu

In [1]:

from pathlib import Path
import os
import re
import sys

import pandas as pd
from IPython.display import Markdown, display

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "dataset").exists()), Path.cwd())
DATA_DIR = Path(os.environ.get("DATATHON_DATA_DIR", project_root / "dataset")).expanduser()
if not DATA_DIR.exists():
    DATA_DIR = Path("./dataset")
if not DATA_DIR.exists():
    raise FileNotFoundError("Dataset folder not found. Set DATATHON_DATA_DIR or place data in ./dataset.")

ARTIFACT_DIR = project_root / "artifacts" / "mcq"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLUMNS = {
    "customers": ["signup_date"],
    "inventory": ["snapshot_date"],
    "orders": ["order_date"],
    "promotions": ["start_date", "end_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "submission": ["Date"],
    "shipments": ["ship_date", "delivery_date"],
    "web_traffic": ["date"],
}

TEXT_COLUMNS_TO_NORMALIZE = {
    "customers": ["gender", "age_group", "acquisition_channel", "city"],
    "geography": ["city", "region", "district"],
    "orders": ["order_status", "payment_method", "device_type", "order_source"],
    "payments": ["payment_method"],
    "products": ["product_name", "category", "segment", "size", "color"],
    "promotions": ["promo_name", "promo_type", "applicable_category", "promo_channel"],
    "returns": ["return_reason"],
    "reviews": ["review_title"],
    "web_traffic": ["traffic_source"],
}


def normalize_text(value):
    if pd.isna(value):
        return value
    text = str(value).strip()
    text = re.sub(r"[\u2010\u2011\u2012\u2013\u2014\u2212]", "-", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()


def normalize_text_columns(df, table):
    out = df.copy()
    for col in TEXT_COLUMNS_TO_NORMALIZE.get(table, []):
        if col in out.columns:
            out[col] = out[col].map(normalize_text)
    return out


def normalize_answer_key(value):
    return normalize_text(value)


dfs = {}
for path in sorted(DATA_DIR.glob("*.csv")):
    table = path.stem
    header = pd.read_csv(path, nrows=0).columns
    parse_dates = [col for col in DATE_COLUMNS.get(table, []) if col in header]
    dfs[table] = normalize_text_columns(pd.read_csv(path, parse_dates=parse_dates, low_memory=False), table)

answers = []


def choose_closest(value, options):
    return min(options, key=lambda key: abs(value - options[key]))


def answer_from_value(value, options):
    normalized_options = {normalize_answer_key(option_value): key for key, option_value in options.items()}
    return normalized_options.get(normalize_answer_key(value))


def record_answer(question, answer, value, evidence):
    answers.append({
        "question": question,
        "answer": answer,
        "computed_value": value,
        "evidence": evidence,
    })
    display(Markdown(f"**Final answer: {answer}**"))

print(f"Loaded {len(dfs)} tables from {DATA_DIR.resolve()}")


Loaded 15 tables from C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\dataset


## Q1. Khoảng Cách Giữa Các Đơn Hàng (Inter-Order Gap)

**Câu hỏi:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)

**A) 30 ngày** | **B) 90 ngày** | **C) 144 ngày** | **D) 365 ngày**

**Logic:** Sắp xếp đơn hàng theo `customer_id` và `order_date`, tính khoảng cách so với đơn hàng trước đó trong cùng một khách hàng, chỉ giữ các khách hàng có nhiều hơn một đơn hàng, sau đó tính trung vị.

In [2]:
orders = dfs["orders"].sort_values(["customer_id", "order_date"]).copy()
orders["prev_order_date"] = orders.groupby("customer_id")["order_date"].shift(1)
orders["gap_days"] = (orders["order_date"] - orders["prev_order_date"]).dt.days
multi_order_customer_ids = orders["customer_id"].value_counts().loc[lambda s: s > 1].index
gap_series = orders.loc[orders["customer_id"].isin(multi_order_customer_ids), "gap_days"].dropna()
median_gap = float(gap_series.median())

display(gap_series.describe().to_frame("gap_days"))

q1_options = {"A": 30, "B": 90, "C": 180, "D": 365}
q1_answer = choose_closest(median_gap, q1_options)
record_answer("Q1", q1_answer, f"median_gap={median_gap:.1f} days", "closest to 180 days")


,gap_days
count,556699.000000
mean,285.592509
std,389.691558
min,0.000000
25%,46.000000
50%,144.000000
75%,357.000000
max,3785.000000


**Final answer: C**

## Q2. Tỷ Suất Lợi Nhuận Gộp Theo Phân Khúc

**Câu hỏi:** Phân khúc sản phẩm (`segment`) nào trong `products.csv` có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức `(price − cogs) / price`?

**A) Premium** | **B) Performance** | **C) Activewear** | **D) Standard**

**Logic:** Tính tỷ suất lợi nhuận gộp cho từng sản phẩm, sau đó tính trung bình theo `segment`.

In [3]:

products = dfs["products"].copy()
products["gross_margin_ratio"] = (products["price"] - products["cogs"]) / products["price"]
segment_margin = (
    products.groupby("segment", as_index=False)["gross_margin_ratio"]
    .mean()
    .sort_values("gross_margin_ratio", ascending=False)
)
display(segment_margin)

top_segment = segment_margin.iloc[0]["segment"]
q2_options = {"A": "premium", "B": "performance", "C": "activewear", "D": "standard"}
q2_answer = answer_from_value(top_segment, q2_options)
record_answer("Q2", q2_answer, top_segment, f"highest average gross margin={segment_margin.iloc[0]['gross_margin_ratio']:.4f}")


,segment,gross_margin_ratio
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


**Final answer: D**

## Q3. Lý Do Trả Hàng Phổ Biến Nhất Của Danh Mục Streetwear

**Câu hỏi:** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join `returns` với `products` theo `product_id`), lý do trả hàng nào xuất hiện nhiều nhất?

**A) defective** | **B) wrong_size** | **C) changed_mind** | **D) not_as_described**

**Logic:** Kết nối `returns` với `products` theo `product_id`, lọc `category == 'streetwear'`, và đếm `return_reason`.

In [4]:

streetwear_returns = (
    dfs["returns"][["return_id", "order_id", "product_id", "return_reason"]]
    .merge(dfs["products"][["product_id", "category"]], on="product_id", how="left")
    .query("category == 'streetwear'")
)
reason_counts = streetwear_returns["return_reason"].value_counts().rename_axis("return_reason").reset_index(name="records")
display(reason_counts)

top_reason = reason_counts.iloc[0]["return_reason"]
q3_options = {"A": "defective", "B": "wrong_size", "C": "changed_mind", "D": "not_as_described"}
q3_answer = answer_from_value(top_reason, q3_options)
record_answer("Q3", q3_answer, top_reason, f"records={int(reason_counts.iloc[0]['records'])}")


,return_reason,records
0,wrong_size,7626
1,defective,4330
2,not_as_described,3854
3,changed_mind,3830
4,late_delivery,2159


**Final answer: B**

## Q4. Tỷ Lệ Thoát Trung Bình Thấp Nhất Theo Nguồn Truy Cập

**Câu hỏi:** Trong `web_traffic.csv`, nguồn truy cập (`traffic_source`) nào có tỷ lệ thoát trung bình (`bounce_rate`) thấp nhất trên tất cả các ngày xuất hiện nguồn đó?

**A) organic_search** | **B) paid_search** | **C) email_campaign** | **D) social_media**

**Logic:** Nhóm theo `traffic_source`, tính trung bình `bounce_rate`, và sắp xếp tăng dần.

In [5]:
source_bounce = (
    dfs["web_traffic"].groupby("traffic_source", as_index=False)["bounce_rate"]
    .mean()
    .sort_values("bounce_rate", ascending=True)
)
display(source_bounce)

best_source = source_bounce.iloc[0]["traffic_source"]
q4_options = {"A": "organic_search", "B": "paid_search", "C": "email_campaign", "D": "social_media"}
q4_answer = answer_from_value(best_source, q4_options)
record_answer("Q4", q4_answer, best_source, f"average bounce_rate={source_bounce.iloc[0]['bounce_rate']:.6f}")


,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511


**Final answer: C**

## Q5. Tỷ Lệ Sử Dụng Khuyến Mãi Trong Order Items

**Câu hỏi:** Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` không null) xấp xỉ là bao nhiêu?

**A) 12%** | **B) 25%** | **C) 39%** | **D) 54%**

**Logic:** Tính trung bình của `promo_id.notna()` nhân 100, sau đó chọn đáp án gần nhất.

In [6]:
order_items = dfs["order_items"].copy()
promo_rate = float(order_items["promo_id"].notna().mean() * 100)
check_table = pd.DataFrame({
    "metric": ["rows", "rows_with_promo_id", "promo_rate_pct"],
    "value": [len(order_items), int(order_items["promo_id"].notna().sum()), promo_rate],
})
display(check_table)

q5_options = {"A": 12, "B": 25, "C": 39, "D": 54}
q5_answer = choose_closest(promo_rate, q5_options)
record_answer("Q5", q5_answer, f"{promo_rate:.2f}%", "closest to 39%")


,metric,value
0,rows,714669.000000
1,rows_with_promo_id,276316.000000
2,promo_rate_pct,38.663493


**Final answer: C**

## Q6. Số Đơn Hàng Trung Bình Theo Nhóm Tuổi

**Câu hỏi:** Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

**A) 55+** | **B) 25–34** | **C) 35–44** | **D) 45–54**

**Logic:** Đếm số đơn hàng theo khách hàng, left join kết quả vào bảng khách hàng có age group, gán 0 cho khách hàng chưa có đơn hàng, sau đó tính trung bình theo nhóm tuổi.

In [7]:

customers = dfs["customers"].copy()
orders = dfs["orders"].copy()
orders_per_customer = orders.groupby("customer_id", as_index=False).size().rename(columns={"size": "num_orders"})
q6_df = customers[customers["age_group"].notna()].merge(orders_per_customer, on="customer_id", how="left")
q6_df["num_orders"] = q6_df["num_orders"].fillna(0)
age_group_stats = (
    q6_df.groupby("age_group", as_index=False)
    .agg(customers=("customer_id", "nunique"), total_orders=("num_orders", "sum"), avg_orders_per_customer=("num_orders", "mean"))
    .sort_values("avg_orders_per_customer", ascending=False)
)
display(age_group_stats)

top_age_group = str(age_group_stats.iloc[0]["age_group"])
q6_options = {"A": "55+", "B": "25-34", "C": "35-44", "D": "45-54"}
q6_answer = answer_from_value(top_age_group, q6_options)
record_answer("Q6", q6_answer, top_age_group, f"avg_orders_per_customer={age_group_stats.iloc[0]['avg_orders_per_customer']:.4f}")


,age_group,customers,total_orders,avg_orders_per_customer
4,55+,13457,72760.0,5.406851
3,45-54,23172,124138.0,5.357241
2,35-44,31920,170368.0,5.337343
1,25-34,36342,190622.0,5.245226
0,18-24,17039,89057.0,5.226656


**Final answer: A**

## Q7. Vùng Có Tổng Doanh Thu Cao Nhất

**Câu hỏi:** Vùng (`region`) nào trong `geography.csv` tạo ra tổng doanh thu cao nhất trong `sales_train.csv`?

**A) West** | **B) Central** | **C) East** | **D) Cả ba vùng có doanh thu xấp xỉ bằng nhau**

**Logic:** `sales.csv` chỉ có `Date`, `Revenue` và `COGS` nên không thể join trực tiếp với vùng. Để trả lời ở cấp độ vùng, notebook sử dụng dữ liệu giao dịch: `order_items -> orders -> geography`, tính doanh thu từng dòng bằng `quantity * unit_price`.

In [ ]:

region_lines = (
    dfs["order_items"][["order_id", "product_id", "quantity", "unit_price"]]
    .merge(dfs["orders"][["order_id", "zip"]], on="order_id", how="left")
    .merge(dfs["geography"][["zip", "region"]], on="zip", how="left")
)

# Q7 wording references sales_train/sales revenue by region, but sales.csv has no region.
region_lines["line_revenue"] = region_lines["quantity"] * region_lines["unit_price"]
region_revenue = (
    region_lines.groupby("region", as_index=False)
    .agg(line_revenue=("line_revenue", "sum"), order_item_rows=("order_id", "size"))
    .sort_values("line_revenue", ascending=False)
)
region_revenue["revenue_share"] = region_revenue["line_revenue"] / region_revenue["line_revenue"].sum()
display(region_revenue)

top_region = region_revenue.iloc[0]["region"]
second_region = region_revenue.iloc[1]["region"]
relative_gap = (region_revenue.iloc[0]["line_revenue"] - region_revenue.iloc[1]["line_revenue"]) / region_revenue.iloc[0]["line_revenue"]

q7_options = {"A": "west", "B": "central", "C": "east", "D": "approximately equal"}
q7_answer = answer_from_value(top_region, q7_options)
record_answer(
    "Q7",
    q7_answer,
    top_region,
    (
        "transaction-level revenue proxy = sum(quantity * unit_price); "
        f"top={top_region}, second={second_region}, relative_gap={relative_gap:.2%}, "
        f"line_revenue={region_revenue.iloc[0]['line_revenue']:.2f}"
    ),
)


,region,line_revenue,line_revenue_after_discount_check,order_item_rows
1,East,7.637533e+09,7.291151e+09,321293
0,Central,4.941908e+09,4.719491e+09,201342
2,West,3.851035e+09,3.670227e+09,192034


**Final answer: C**

## Q8. Phương Thức Thanh Toán Phổ Biến Nhất Trong Đơn Hàng Bị Huỷ

**Câu hỏi:** Trong các đơn hàng có `order_status = 'cancelled'` trong `orders.csv`, phương thức thanh toán nào được sử dụng nhiều nhất?

**A) credit_card** | **B) cod** | **C) paypal** | **D) bank_transfer**

**Logic:** Lọc các đơn hàng bị huỷ và đếm theo `payment_method`.

In [9]:
cancelled = dfs["orders"].query("order_status == 'cancelled'").copy()
payment_counts = cancelled["payment_method"].value_counts().rename_axis("payment_method").reset_index(name="orders")
display(payment_counts)

top_payment = payment_counts.iloc[0]["payment_method"]
q8_options = {"A": "credit_card", "B": "cod", "C": "paypal", "D": "bank_transfer"}
q8_answer = answer_from_value(top_payment, q8_options)
record_answer("Q8", q8_answer, top_payment, f"cancelled_orders={int(payment_counts.iloc[0]['orders'])}")


,payment_method,orders
0,credit_card,28452
1,cod,15468
2,paypal,7817
3,apple_pay,5190
4,bank_transfer,2535


**Final answer: A**

## Q9. Tỷ Lệ Trả Hàng Cao Nhất Theo Kích Cỡ

**Câu hỏi:** Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (sau khi join với `products` theo `product_id`)?

**A) S** | **B) M** | **C) L** | **D) XL**

**Logic:** Mẫu số là số dòng order-item theo kích cỡ; tử số là số bản ghi trả hàng theo kích cỡ. Tính tỷ lệ và so sánh bốn kích cỡ được yêu cầu.

In [10]:

prod_size = dfs["products"][["product_id", "size"]].copy()
order_items_size = dfs["order_items"][["order_id", "product_id"]].merge(prod_size, on="product_id", how="left")
returns_size = dfs["returns"][["return_id", "product_id"]].merge(prod_size, on="product_id", how="left")

denom = order_items_size.groupby("size").size().reset_index(name="order_item_rows")
numer = returns_size.groupby("size").size().reset_index(name="return_records")
q9_stats = denom.merge(numer, on="size", how="left").fillna({"return_records": 0})
q9_stats = q9_stats[q9_stats["size"].isin(["s", "m", "l", "xl"])].copy()
q9_stats["return_rate"] = q9_stats["return_records"] / q9_stats["order_item_rows"]
q9_stats = q9_stats.sort_values("return_rate", ascending=False)
display(q9_stats)

top_size = q9_stats.iloc[0]["size"]
q9_options = {"A": "s", "B": "m", "C": "l", "D": "xl"}
q9_answer = answer_from_value(top_size, q9_options)
record_answer("Q9", q9_answer, top_size, f"return_rate={q9_stats.iloc[0]['return_rate']:.6f}")


,size,order_item_rows,return_records,return_rate
2,S,172042,9723,0.056515
0,L,173174,9741,0.056250
1,M,176428,9820,0.055660
3,XL,193025,10655,0.055200


**Final answer: A**

## Q10. Kế Hoạch Trả Góp Có Giá Trị Thanh Toán Trung Bình Cao Nhất

**Câu hỏi:** Trong `payments.csv`, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

**A) 1 kỳ (trả một lần)** | **B) 3 kỳ** | **C) 6 kỳ** | **D) 12 kỳ**

**Logic:** Nhóm theo `installments`, tính trung bình `payment_value`, và chọn lựa chọn cao nhất trong số các đáp án.

In [11]:
installment_stats = (
    dfs["payments"].groupby("installments", as_index=False)["payment_value"]
    .mean()
    .sort_values("payment_value", ascending=False)
)
display(installment_stats)

q10_options = {"A": 1, "B": 3, "C": 6, "D": 12}
option_stats = installment_stats[installment_stats["installments"].isin(q10_options.values())].copy()
best_installments = int(option_stats.iloc[0]["installments"])
q10_answer = answer_from_value(best_installments, q10_options)
record_answer("Q10", q10_answer, f"{best_installments} installments", f"avg_payment_value={option_stats.iloc[0]['payment_value']:.2f}")


,installments,payment_value
3,6,24446.654403
2,3,24399.635486
4,12,24245.772694
0,1,24113.274166
1,2,708.473729


**Final answer: C**

## Bảng Tổng Hợp Đáp Án

In [12]:

answers_df = pd.DataFrame(answers)
answers_df["answer_only"] = answers_df["answer"].str.extract(r"([A-D])", expand=False)

expected_questions = [f"Q{i}" for i in range(1, 11)]
assert len(answers_df) == 10, f"Expected 10 answers, got {len(answers_df)}"
assert answers_df["answer"].notna().all(), "Some MCQ answers are missing. Check answer_from_value matching."
assert answers_df["question"].tolist() == expected_questions, f"Unexpected question order: {answers_df['question'].tolist()}"
assert answers_df["answer_only"].notna().all(), "Some answers are not valid A-D labels."

display(answers_df)

out_path = ARTIFACT_DIR / "mcq_answers.csv"
answers_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved", out_path.relative_to(project_root))


,question,answer,computed_value,evidence,answer_only
0,Q1,C,median_gap=144.0 days,closest to 180 days,C
1,Q2,D,Standard,highest average gross margin=0.3134,D
2,Q3,B,wrong_size,records=7626,B
3,Q4,C,email_campaign,average bounce_rate=0.004458,C
4,Q5,C,38.66%,closest to 39%,C
5,Q6,A,55+,avg_orders_per_customer=5.4069,A
6,Q7,C,East,line_revenue=7637532676.20,C
7,Q8,A,credit_card,cancelled_orders=28452,A
8,Q9,A,S,return_rate=0.056515,A
9,Q10,C,6 installments,avg_payment_value=24446.65,C


Saved artifacts\mcq\mcq_answers.csv
